In [11]:
#IMPLEMENT SIMPLE REINFORCE
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import gymnasium as gym

In [12]:
#create the enviroment
env = gym.make("CartPole-v1")

In [19]:
#create policy network
model = tf.keras.Sequential([
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(2, activation="softmax")
])

In [14]:
#create the optimizer(optimizer changes weights)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.01)

In [15]:
#RL parameters
episodes = 500
discount_factor = 0.99


In [27]:
#Train using reinforce(the difference between reinforce and policy gradient is like model.fit(x_train...) and model=linearRregression)
for episode in range(episodes):

  state, info = env.reset()#reset to the initial state

  rewards = [] #store the reward the agent recieves

  states = [] #store the states of the agent in each episode

  actions = [] #store the actions the agents takes

  done = False

  while not done:

    states.append(state)#remebers the current state

    probabilities = model(np.array([state]), training=False)[0]#gives the current state to NN, the network retuns probabilities for every possible avtion

    probs = probabilities.numpy().astype("float64")#convert tensor to numpy cause it kept poopping an error

    probs /= np.sum(probs)#normalize the probabilities

    action = np.random.choice(2, p=probs)#choose an action using the probability from up

    actions.append(action)

    state, reward, terminated, truncated, info = env.step(action)

    rewards.append(reward)

    done = terminated or truncated

  returns = []

  total = 0

  for reward in reversed(rewards):#rewards affect the value of earlier action

    total = reward + discount_factor * total#dicount returns(its different from discount factor)

    returns.append(total)

  returns.reverse()#we callculated returns backward

  with tf.GradientTape() as tape:#tensorflow calculates the gradients that the optimizer uses to change weights

     loss = 0

     for state, action, total_reward in zip(states, actions, returns): #go through each state and its return

         probabilities = model(np.array([state]), training=True)[0]#asking for action probabilities again but tensorflow is tracking

         log_probability = tf.math.log(probabilities[action])#get the probability of the action that was taken

         loss -= log_probability * total_reward#adjust loss according to how good the probability was

  gradients = tape.gradient(loss, model.trainable_variables)#calculate how much each model weight contributed to the loss

  optimizer.apply_gradients(zip(gradients, model.trainable_variables))#use the gradient to update weights

  if episode % 50 == 0:
    print("episode:", episode, "Reward:", sum(rewards))









episode: 0 Reward: 15.0
episode: 50 Reward: 74.0
episode: 100 Reward: 191.0
episode: 150 Reward: 154.0
episode: 200 Reward: 135.0
episode: 250 Reward: 291.0
episode: 300 Reward: 88.0
episode: 350 Reward: 500.0
episode: 400 Reward: 18.0
episode: 450 Reward: 281.0


In [26]:
state, info = env.reset()

probabilities = model(np.array([state]), training=False)[0]

print("probabilites:", probabilities)

probabilites: tf.Tensor([0.5011126  0.49888736], shape=(2,), dtype=float32)
